In [9]:
import pandas as pd

# Load both files
df_metrics = pd.read_parquet('/content/all_metrics_NORMALIZED.parquet')
df_labels = pd.read_csv('/content/behavior_labels (1).csv')

print(f"Metrics: {len(df_metrics)} rows, {df_metrics['id'].nunique()} unique IDs")
print(f"Labels: {len(df_labels)} rows, {df_labels['id'].nunique()} unique IDs")

# Check column names in labels file
print(f"\nLabel file columns: {df_labels.columns.tolist()}")

# Rename 'label' to 'behavior_label' if needed
if 'label' in df_labels.columns and 'behavior_label' not in df_labels.columns:
    df_labels['behavior_label'] = df_labels['label']

# Merge on 'id'
df_final = df_metrics.merge(
    df_labels[['id', 'behavior_label']],
    on='id',
    how='left'
)

# Check results
print(f"\n✅ Merged: {len(df_final)} rows")
print(f"Missing labels: {df_final['behavior_label'].isna().sum()}")
print(f"\nBehavior label distribution:")
print(df_final['behavior_label'].value_counts())

# Save
df_final.to_parquet('/content/all_metrics_WITH_LABELS.parquet')
print("\n✅ Saved: all_metrics_WITH_LABELS.parquet")

Metrics: 280896 rows, 266 unique IDs
Labels: 300 rows, 300 unique IDs

Label file columns: ['id', 'dataset', 'constraint_tags', 'assistant_generated', 'label', 'length_ok', 'keyword_ok', 'json_ok', 'no_explanations_ok', 'tone_concise_ok', 'reasons']

✅ Merged: 280896 rows
Missing labels: 31680

Behavior label distribution:
behavior_label
1.0    159456
0.0     89760
Name: count, dtype: int64

✅ Saved: all_metrics_WITH_LABELS.parquet


In [11]:
"""
COMPLETE WEEK 4 ANALYSIS WITH REAL BEHAVIOR LABELS
Run this script in Google Colab after uploading:
1. all_metrics_NORMALIZED.parquet
2. behavior_labels.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from scipy.spatial.distance import cosine
import warnings
import os
warnings.filterwarnings('ignore')

print("="*80)
print("WEEK 4 ANALYSIS WITH REAL BEHAVIOR LABELS")
print("="*80)

# ============================================================================
# STEP 1: LOAD AND MERGE DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING AND MERGING DATA")
print("="*80)

# Load attention metrics
print("\nLoading all_metrics_NORMALIZED.parquet...")
df_metrics = pd.read_parquet('/content/all_metrics_NORMALIZED.parquet')
print(f"✅ Loaded {len(df_metrics)} rows")
print(f"   Unique IDs: {df_metrics['id'].nunique()}")
print(f"   Datasets: {df_metrics['dataset'].unique()}")

# Load behavior labels
print("\nLoading behavior_labels.csv...")
df_labels = pd.read_csv('/content/behavior_labels (1).csv')
print(f"✅ Loaded {len(df_labels)} rows")
print(f"   Columns: {df_labels.columns.tolist()}")
print(f"   Unique IDs: {df_labels['id'].nunique()}")

# Rename 'label' to 'behavior_label'
if 'label' in df_labels.columns:
    df_labels['behavior_label'] = df_labels['label']
    print("   Renamed 'label' → 'behavior_label'")

# Check label distribution
print(f"\nBehavior label distribution:")
print(df_labels['behavior_label'].value_counts())
print(f"Compliance rate: {df_labels['behavior_label'].mean():.3f}")

# Merge
print("\nMerging datasets on 'id'...")
df_final = df_metrics.merge(
    df_labels[['id', 'behavior_label']],
    on='id',
    how='left'
)

print(f"✅ Merged successfully: {len(df_final)} rows")

# Check for missing labels
missing = df_final['behavior_label'].isna().sum()
if missing > 0:
    print(f"⚠️  WARNING: {missing} rows have missing behavior_label")
    print(f"   These IDs are in metrics but not in labels")
    print(f"   They will be dropped from analysis")
    df_final = df_final.dropna(subset=['behavior_label'])
    print(f"   After dropping: {len(df_final)} rows")

print(f"\nFinal dataset:")
print(f"  Rows: {len(df_final)}")
print(f"  Unique examples: {df_final['id'].nunique()}")
print(f"  Datasets: {df_final['dataset'].unique()}")
print(f"  Behavior label distribution:")
print(df_final['behavior_label'].value_counts())

# ============================================================================
# STEP 2: CLEAN DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 2: CLEANING DATA")
print("="*80)

initial_size = len(df_final)

# Remove 'avg' heads
df_final = df_final[df_final['head'] != 'avg']
print(f"Removed 'avg' heads: {len(df_final)} rows remaining")

# Ensure correct types
df_final['head'] = pd.to_numeric(df_final['head'], errors='coerce')
df_final = df_final.dropna(subset=['head'])
df_final['head'] = df_final['head'].astype(int)
print(f"Converted head to int: {len(df_final)} rows")

df_final['layer'] = pd.to_numeric(df_final['layer'], errors='coerce')
df_final = df_final.dropna(subset=['layer'])
df_final['layer'] = df_final['layer'].astype(int)
print(f"Converted layer to int: {len(df_final)} rows")

df_final['behavior_label'] = df_final['behavior_label'].astype(int)
print(f"Converted behavior_label to int")

print(f"\n✅ Clean dataset ready:")
print(f"   Total rows: {len(df_final)}")
print(f"   Layers: {df_final['layer'].min()}-{df_final['layer'].max()}")
print(f"   Heads per layer: {df_final['head'].nunique()}")

# ============================================================================
# STEP 3: COMPUTE PER-HEAD CORRELATIONS
# ============================================================================

print("\n" + "="*80)
print("STEP 3: COMPUTING PER-HEAD CORRELATIONS")
print("="*80)

datasets = df_final['dataset'].unique()
correlation_results = {}

for dataset in datasets:
    print(f"\n{dataset.upper()}:")
    df_dataset = df_final[df_final['dataset'] == dataset]

    print(f"  Processing {len(df_dataset)} rows...")
    print(f"  Unique examples: {df_dataset['id'].nunique()}")
    print(f"  Compliance rate: {df_dataset['behavior_label'].mean():.3f}")

    # Group by (layer, head)
    grouped = df_dataset.groupby(['layer', 'head'])

    correlations = []
    processed = 0

    for (layer, head), group in grouped:
        if len(group) < 2:  # Need at least 2 samples
            continue

        corr_data = {
            'layer': layer,
            'head': head,
            'n_samples': len(group)
        }

        # Compute correlations for each metric
        for metric in ['PAM', 'QAM', 'SAM']:
            col = f'{metric}_norm'
            try:
                corr, pval = pearsonr(group[col], group['behavior_label'])
                corr_data[f'{metric}_corr'] = corr
                corr_data[f'{metric}_pval'] = pval
            except:
                corr_data[f'{metric}_corr'] = np.nan
                corr_data[f'{metric}_pval'] = np.nan

        correlations.append(corr_data)
        processed += 1

    # Create DataFrame
    corr_df = pd.DataFrame(correlations)
    corr_df = corr_df.sort_values('PAM_corr', ascending=False)

    correlation_results[dataset] = corr_df

    # Print summary
    print(f"  ✅ Computed {len(corr_df)} head correlations")
    print(f"     PAM corr range: [{corr_df['PAM_corr'].min():.3f}, {corr_df['PAM_corr'].max():.3f}]")
    print(f"     QAM corr range: [{corr_df['QAM_corr'].min():.3f}, {corr_df['QAM_corr'].max():.3f}]")
    print(f"     SAM corr range: [{corr_df['SAM_corr'].min():.3f}, {corr_df['SAM_corr'].max():.3f}]")

# ============================================================================
# STEP 4: IDENTIFY TOP HEADS
# ============================================================================

print("\n" + "="*80)
print("STEP 4: IDENTIFYING TOP HEADS")
print("="*80)

top_heads = {}

for dataset, corr_df in correlation_results.items():
    print(f"\n{dataset.upper()}:")

    # Top 10 PAM heads (positive correlation = prompt-following)
    top_pam = corr_df.nlargest(10, 'PAM_corr')[['layer', 'head', 'PAM_corr', 'PAM_pval']]
    print(f"\n  Top 10 Positive PAM Heads (Prompt-Following):")
    for idx, row in top_pam.iterrows():
        sig = "***" if row['PAM_pval'] < 0.001 else "**" if row['PAM_pval'] < 0.01 else "*" if row['PAM_pval'] < 0.05 else ""
        print(f"    Layer {int(row['layer'])}, Head {int(row['head'])}: {row['PAM_corr']:.4f} (p={row['PAM_pval']:.4f}) {sig}")

    # Bottom 10 PAM heads (negative correlation)
    bottom_pam = corr_df.nsmallest(10, 'PAM_corr')[['layer', 'head', 'PAM_corr', 'PAM_pval']]
    print(f"\n  Top 10 Negative PAM Heads:")
    for idx, row in bottom_pam.head(5).iterrows():
        sig = "***" if row['PAM_pval'] < 0.001 else "**" if row['PAM_pval'] < 0.01 else "*" if row['PAM_pval'] < 0.05 else ""
        print(f"    Layer {int(row['layer'])}, Head {int(row['head'])}: {row['PAM_corr']:.4f} (p={row['PAM_pval']:.4f}) {sig}")

    # Top SAM heads
    top_sam = corr_df.nlargest(10, 'SAM_corr')[['layer', 'head', 'SAM_corr', 'SAM_pval']]
    print(f"\n  Top 10 Positive SAM Heads (Self-Attention):")
    for idx, row in top_sam.head(5).iterrows():
        sig = "***" if row['SAM_pval'] < 0.001 else "**" if row['SAM_pval'] < 0.01 else "*" if row['SAM_pval'] < 0.05 else ""
        print(f"    Layer {int(row['layer'])}, Head {int(row['head'])}: {row['SAM_corr']:.4f} (p={row['SAM_pval']:.4f}) {sig}")

    top_heads[dataset] = {
        'top_pam_positive': top_pam,
        'top_pam_negative': bottom_pam,
        'top_sam_positive': top_sam
    }

# ============================================================================
# STEP 5: SAVE RESULTS
# ============================================================================

print("\n" + "="*80)
print("STEP 5: SAVING RESULTS")
print("="*80)

output_dir = './week4_outputs'
os.makedirs(output_dir, exist_ok=True)

# Save correlation tables
for dataset, corr_df in correlation_results.items():
    filename = f"{output_dir}/head_corr_{dataset}.csv"
    corr_df.to_csv(filename, index=False)
    print(f"✅ Saved: {filename}")

# Save top heads summary
summary_file = f"{output_dir}/top_heads_summary.txt"
with open(summary_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TOP HEADS SUMMARY - WEEK 4 ANALYSIS\n")
    f.write("="*80 + "\n\n")

    for dataset, tops in top_heads.items():
        f.write(f"\n{'='*80}\n")
        f.write(f"{dataset.upper()}\n")
        f.write(f"{'='*80}\n\n")

        f.write("Top 10 Positive PAM Heads (Prompt-Following):\n")
        f.write(tops['top_pam_positive'].to_string(index=False))
        f.write("\n\n")

        f.write("Top 10 Negative PAM Heads:\n")
        f.write(tops['top_pam_negative'].to_string(index=False))
        f.write("\n\n")

        f.write("Top 10 Positive SAM Heads (Self-Attention):\n")
        f.write(tops['top_sam_positive'].to_string(index=False))
        f.write("\n\n")

print(f"✅ Saved: {summary_file}")

# ============================================================================
# STEP 6: CREATE VISUALIZATIONS
# ============================================================================

print("\n" + "="*80)
print("STEP 6: CREATING VISUALIZATIONS")
print("="*80)

# Create correlation heatmaps
for dataset, corr_df in correlation_results.items():
    for metric in ['PAM_corr', 'QAM_corr', 'SAM_corr']:
        # Create pivot table
        pivot = corr_df.pivot(index='layer', columns='head', values=metric)

        # Create heatmap
        plt.figure(figsize=(16, 8))
        sns.heatmap(pivot, annot=False, cmap='RdBu_r', center=0,
                   vmin=-0.5, vmax=0.5, cbar_kws={'label': 'Correlation'})
        plt.title(f'{dataset}: {metric.replace("_", " ").title()} by Layer and Head')
        plt.xlabel('Head')
        plt.ylabel('Layer')
        plt.tight_layout()

        filename = f"{output_dir}/heatmap_{dataset}_{metric}.png"
        plt.savefig(filename, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"✅ Saved: {filename}")

# Create distribution plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, metric in enumerate(['PAM_corr', 'QAM_corr', 'SAM_corr']):
    ax = axes[idx]
    for dataset, corr_df in correlation_results.items():
        ax.hist(corr_df[metric].dropna(), bins=30, alpha=0.5, label=dataset)
    ax.set_xlabel('Correlation')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{metric.replace("_", " ").title()} Distribution')
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
filename = f"{output_dir}/correlation_distributions.png"
plt.savefig(filename, dpi=200, bbox_inches='tight')
plt.close()
print(f"✅ Saved: {filename}")

# ============================================================================
# STEP 7: GENERATE INTERPRETATION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: GENERATING INTERPRETATION")
print("="*80)

interp_file = f"{output_dir}/analysis_interpretation.txt"
with open(interp_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("WEEK 4 ANALYSIS INTERPRETATION\n")
    f.write("="*80 + "\n\n")

    f.write("DATASET OVERVIEW:\n")
    f.write("-"*80 + "\n")
    for dataset, corr_df in correlation_results.items():
        f.write(f"\n{dataset.upper()}:\n")
        f.write(f"  Number of heads analyzed: {len(corr_df)}\n")
        f.write(f"  Mean PAM correlation: {corr_df['PAM_corr'].mean():.4f} (std: {corr_df['PAM_corr'].std():.4f})\n")
        f.write(f"  Mean QAM correlation: {corr_df['QAM_corr'].mean():.4f} (std: {corr_df['QAM_corr'].std():.4f})\n")
        f.write(f"  Mean SAM correlation: {corr_df['SAM_corr'].mean():.4f} (std: {corr_df['SAM_corr'].std():.4f})\n")

        # Count significant correlations
        sig_pam = (corr_df['PAM_pval'] < 0.05).sum()
        f.write(f"  Significant PAM correlations (p<0.05): {sig_pam}/{len(corr_df)} ({sig_pam/len(corr_df)*100:.1f}%)\n")

    f.write("\n" + "="*80 + "\n")
    f.write("KEY FINDINGS:\n")
    f.write("="*80 + "\n\n")

    # Find dataset with strongest signal
    best_dataset = None
    best_signal = 0
    for dataset, corr_df in correlation_results.items():
        signal = abs(corr_df['PAM_corr'].mean())
        if signal > best_signal:
            best_signal = signal
            best_dataset = dataset

    if best_signal > 0.1:
        f.write(f"✓ STRONG SIGNAL DETECTED in {best_dataset}\n")
        f.write(f"  Mean PAM correlation: {correlation_results[best_dataset]['PAM_corr'].mean():.4f}\n")
        f.write(f"  This dataset shows clear attention → behavior relationship\n\n")
    elif best_signal > 0.05:
        f.write(f"~ MODERATE SIGNAL in {best_dataset}\n")
        f.write(f"  Mean PAM correlation: {best_signal:.4f}\n")
        f.write(f"  Some relationship between attention and behavior detected\n\n")
    else:
        f.write(f"⚠ WEAK SIGNAL across all datasets\n")
        f.write(f"  Strongest mean correlation: {best_signal:.4f}\n")
        f.write(f"  Attention patterns may not strongly predict behavior for this model\n\n")

    f.write("\n" + "="*80 + "\n")
    f.write("RECOMMENDATIONS FOR WEEK 5:\n")
    f.write("="*80 + "\n\n")

    if best_dataset:
        f.write(f"Focus causal experiments on: {best_dataset}\n\n")

        corr_df = correlation_results[best_dataset]
        top_pam = corr_df.nlargest(5, 'PAM_corr')[['layer', 'head', 'PAM_corr']]

        f.write("Recommended heads for PAM boost:\n")
        for _, row in top_pam.iterrows():
            f.write(f"  - Layer {int(row['layer'])}, Head {int(row['head'])} (corr: {row['PAM_corr']:.4f})\n")

print(f"✅ Saved: {interp_file}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("WEEK 4 ANALYSIS COMPLETE!")
print("="*80)
print(f"\nAll outputs saved to: {output_dir}/")
print("\nGenerated files:")
print("  📊 Correlation tables:")
for dataset in correlation_results.keys():
    print(f"     - head_corr_{dataset}.csv")
print("  📈 Visualizations:")
print(f"     - heatmap_*.png (correlation heatmaps)")
print(f"     - correlation_distributions.png")
print("  📝 Summaries:")
print(f"     - top_heads_summary.txt")
print(f"     - analysis_interpretation.txt")

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Review the correlation tables and heatmaps")
print("2. Check analysis_interpretation.txt for key findings")
print("3. Use top heads from top_heads_summary.txt for Week 5 experiments")
print("\n" + "="*80)

# Return results for further analysis
results = {
    'correlation_results': correlation_results,
    'top_heads': top_heads,
    'df': df_final
}

print("\n✅ Analysis complete! Results stored in 'results' variable.")

WEEK 4 ANALYSIS WITH REAL BEHAVIOR LABELS

STEP 1: LOADING AND MERGING DATA

Loading all_metrics_NORMALIZED.parquet...
✅ Loaded 280896 rows
   Unique IDs: 266
   Datasets: ['alpaca' 'flan' 'sharegpt']

Loading behavior_labels.csv...
✅ Loaded 300 rows
   Columns: ['id', 'dataset', 'constraint_tags', 'assistant_generated', 'label', 'length_ok', 'keyword_ok', 'json_ok', 'no_explanations_ok', 'tone_concise_ok', 'reasons']
   Unique IDs: 300
   Renamed 'label' → 'behavior_label'

Behavior label distribution:
behavior_label
1.0    161
0.0     89
Name: count, dtype: int64
Compliance rate: 0.644

Merging datasets on 'id'...
✅ Merged successfully: 280896 rows
⚠️  WARNING: 31680 rows have missing behavior_label
   These IDs are in metrics but not in labels
   They will be dropped from analysis
   After dropping: 249216 rows

Final dataset:
  Rows: 249216
  Unique examples: 236
  Datasets: ['alpaca' 'flan' 'sharegpt']
  Behavior label distribution:
behavior_label
1.0    159456
0.0     89760
Name:

In [6]:
"""
Week 4: Head-Level Analysis - Google Colab Version
Complete pipeline for per-head correlation analysis, feature importance, and cross-dataset comparison

TO USE IN GOOGLE COLAB:
1. Upload this file and your data file to Colab
2. Run:
   from week4_colab_version import run_week4_analysis
   results = run_week4_analysis('/content/all_metrics_WITH_LABELS.parquet')
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# ============================================================================
# RESEARCHER 1: HEAD-LEVEL METRIC COMPUTATION
# ============================================================================

class HeadLevelCorrelations:
    """Compute per-head correlations with behavior labels"""

    def __init__(self, data_path):
        """Load and prepare data"""
        print("="*80)
        print("RESEARCHER 1: HEAD-LEVEL METRIC COMPUTATION")
        print("="*80)

        # Load data
        print(f"\nLoading data from: {data_path}")
        self.df = pd.read_parquet(data_path)
        print(f"Loaded {len(self.df)} rows")
        print(f"Columns: {self.df.columns.tolist()}")

        # Data cleaning
        self._clean_data()

    def _clean_data(self):
        """Clean and prepare the dataset"""
        print("\nCleaning data...")

        # Initial size
        initial_size = len(self.df)

        # Drop rows with missing behavior labels
        self.df = self.df.dropna(subset=['behavior_label'])
        print(f"  - Dropped {initial_size - len(self.df)} rows with missing behavior_label")

        # Filter out 'avg' heads (keep only specific head numbers)
        if 'head' in self.df.columns:
            self.df = self.df[self.df['head'] != 'avg']
            # Ensure head is int
            self.df['head'] = pd.to_numeric(self.df['head'], errors='coerce')
            self.df = self.df.dropna(subset=['head'])
            self.df['head'] = self.df['head'].astype(int)

        # Ensure layer is int
        if 'layer' in self.df.columns:
            self.df['layer'] = pd.to_numeric(self.df['layer'], errors='coerce')
            self.df = self.df.dropna(subset=['layer'])
            self.df['layer'] = self.df['layer'].astype(int)

        print(f"  - Final dataset size: {len(self.df)} rows")
        print(f"  - Unique datasets: {self.df['dataset'].unique() if 'dataset' in self.df.columns else 'N/A'}")
        print(f"  - Behavior label distribution:\n{self.df['behavior_label'].value_counts()}")

    def compute_correlations(self):
        """Compute per-head correlations for each dataset"""
        print("\n" + "-"*80)
        print("Computing per-head correlations...")
        print("-"*80)

        datasets = self.df['dataset'].unique()
        self.correlation_results = {}

        for dataset in datasets:
            print(f"\nProcessing {dataset}...")
            df_dataset = self.df[self.df['dataset'] == dataset]

            # Group by (layer, head)
            grouped = df_dataset.groupby(['layer', 'head'])

            correlations = []

            for (layer, head), group in grouped:
                if len(group) < 2:  # Need at least 2 samples for correlation
                    continue

                # Compute correlations for PAM, QAM, SAM
                corr_data = {
                    'layer': layer,
                    'head': head,
                    'n_samples': len(group)
                }

                # PAM correlation
                if 'PAM_norm' in group.columns:
                    try:
                        corr, pval = pearsonr(group['PAM_norm'], group['behavior_label'])
                        corr_data['PAM_corr'] = corr
                        corr_data['PAM_pval'] = pval
                    except:
                        corr_data['PAM_corr'] = np.nan
                        corr_data['PAM_pval'] = np.nan

                # QAM correlation
                if 'QAM_norm' in group.columns:
                    try:
                        corr, pval = pearsonr(group['QAM_norm'], group['behavior_label'])
                        corr_data['QAM_corr'] = corr
                        corr_data['QAM_pval'] = pval
                    except:
                        corr_data['QAM_corr'] = np.nan
                        corr_data['QAM_pval'] = np.nan

                # SAM correlation
                if 'SAM_norm' in group.columns:
                    try:
                        corr, pval = pearsonr(group['SAM_norm'], group['behavior_label'])
                        corr_data['SAM_corr'] = corr
                        corr_data['SAM_pval'] = pval
                    except:
                        corr_data['SAM_corr'] = np.nan
                        corr_data['SAM_pval'] = np.nan

                correlations.append(corr_data)

            # Create DataFrame
            corr_df = pd.DataFrame(correlations)
            corr_df = corr_df.sort_values('PAM_corr', ascending=False)

            self.correlation_results[dataset] = corr_df

            # Print summary
            print(f"  - Computed correlations for {len(corr_df)} heads")
            print(f"  - PAM correlation range: [{corr_df['PAM_corr'].min():.3f}, {corr_df['PAM_corr'].max():.3f}]")
            print(f"  - QAM correlation range: [{corr_df['QAM_corr'].min():.3f}, {corr_df['QAM_corr'].max():.3f}]")
            print(f"  - SAM correlation range: [{corr_df['SAM_corr'].min():.3f}, {corr_df['SAM_corr'].max():.3f}]")

        return self.correlation_results

    def identify_top_heads(self, n_top=10):
        """Identify top positive and negative heads for each metric"""
        print("\n" + "-"*80)
        print(f"Identifying top {n_top} heads per dataset...")
        print("-"*80)

        self.top_heads = {}

        for dataset, corr_df in self.correlation_results.items():
            print(f"\n{dataset.upper()}:")

            dataset_tops = {}

            # Top PAM heads (positive correlation = prompt-following)
            top_pam_pos = corr_df.nlargest(n_top, 'PAM_corr')[['layer', 'head', 'PAM_corr']]
            print(f"\n  Top {n_top} Positive PAM Heads (Prompt-Following):")
            print(top_pam_pos.to_string(index=False))
            dataset_tops['top_pam_positive'] = top_pam_pos

            # Bottom PAM heads (negative correlation)
            top_pam_neg = corr_df.nsmallest(n_top, 'PAM_corr')[['layer', 'head', 'PAM_corr']]
            print(f"\n  Top {n_top} Negative PAM Heads:")
            print(top_pam_neg.to_string(index=False))
            dataset_tops['top_pam_negative'] = top_pam_neg

            # Top QAM heads
            top_qam_pos = corr_df.nlargest(n_top, 'QAM_corr')[['layer', 'head', 'QAM_corr']]
            print(f"\n  Top {n_top} Positive QAM Heads (User Message Attention):")
            print(top_qam_pos.to_string(index=False))
            dataset_tops['top_qam_positive'] = top_qam_pos

            # Top SAM heads
            top_sam_pos = corr_df.nlargest(n_top, 'SAM_corr')[['layer', 'head', 'SAM_corr']]
            print(f"\n  Top {n_top} Positive SAM Heads (Self-Attention):")
            print(top_sam_pos.to_string(index=False))
            dataset_tops['top_sam_positive'] = top_sam_pos

            top_sam_neg = corr_df.nsmallest(n_top, 'SAM_corr')[['layer', 'head', 'SAM_corr']]
            print(f"\n  Top {n_top} Negative SAM Heads:")
            print(top_sam_neg.to_string(index=False))
            dataset_tops['top_sam_negative'] = top_sam_neg

            self.top_heads[dataset] = dataset_tops

        return self.top_heads

    def save_results(self, output_dir='./week4_outputs'):
        """Save correlation tables to CSV"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n" + "="*80)
        print(f"Saving results to {output_dir}...")
        print("="*80)

        for dataset, corr_df in self.correlation_results.items():
            filename = f"{output_dir}/head_corr_{dataset.lower()}.csv"
            corr_df.to_csv(filename, index=False)
            print(f"  - Saved {filename}")

        # Save top heads summary
        with open(f"{output_dir}/top_heads_summary.txt", 'w') as f:
            for dataset, tops in self.top_heads.items():
                f.write(f"\n{'='*80}\n")
                f.write(f"{dataset.upper()}\n")
                f.write(f"{'='*80}\n")
                for key, df in tops.items():
                    f.write(f"\n{key.replace('_', ' ').title()}:\n")
                    f.write(df.to_string(index=False))
                    f.write("\n")

        print(f"  - Saved {output_dir}/top_heads_summary.txt")


# ============================================================================
# RESEARCHER 2: FEATURE IMPORTANCE & HEAD RANKING
# ============================================================================

class FeatureImportanceAnalysis:
    """Compute feature importance using Logistic Regression and Random Forest"""

    def __init__(self, df, correlation_results):
        """Initialize with cleaned data and correlation results"""
        print("\n\n" + "="*80)
        print("RESEARCHER 2: FEATURE IMPORTANCE & HEAD RANKING")
        print("="*80)

        self.df = df
        self.correlation_results = correlation_results
        self.importance_results = {}

    def compute_feature_importance(self):
        """Train models and extract feature importances"""
        print("\nComputing feature importance for each dataset...")
        print("-"*80)

        datasets = self.df['dataset'].unique()

        for dataset in datasets:
            print(f"\n{dataset.upper()}:")
            df_dataset = self.df[self.df['dataset'] == dataset].copy()

            # Aggregate metrics per example
            if 'example_id' in df_dataset.columns:
                agg_data = df_dataset.groupby('example_id').agg({
                    'PAM_norm': 'mean',
                    'QAM_norm': 'mean',
                    'SAM_norm': 'mean',
                    'behavior_label': 'first'
                }).reset_index()
            else:
                agg_data = df_dataset[['PAM_norm', 'QAM_norm', 'SAM_norm', 'behavior_label']].copy()

            # Remove NaN
            agg_data = agg_data.dropna()

            if len(agg_data) < 10:
                print(f"  - Insufficient data ({len(agg_data)} samples), skipping")
                continue

            # Prepare X and y
            X = agg_data[['PAM_norm', 'QAM_norm', 'SAM_norm']].values
            y = agg_data['behavior_label'].values

            # Check if we have both classes
            if len(np.unique(y)) < 2:
                print(f"  - Only one class present, skipping classification")
                continue

            # Split data
            try:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42, stratify=y
                )
            except:
                X_train, X_test, y_train, y_test = train_test_split(
                    X, y, test_size=0.2, random_state=42
                )

            # Standardize features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Train Logistic Regression
            print(f"  - Training Logistic Regression on {len(X_train)} samples...")
            lr = LogisticRegression(random_state=42, max_iter=1000)
            lr.fit(X_train_scaled, y_train)
            lr_score = lr.score(X_test_scaled, y_test)
            lr_coefs = lr.coef_[0]
            print(f"    Accuracy: {lr_score:.3f}")

            # Train Random Forest
            print(f"  - Training Random Forest...")
            rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
            rf.fit(X_train, y_train)
            rf_score = rf.score(X_test, y_test)
            rf_importances = rf.feature_importances_
            print(f"    Accuracy: {rf_score:.3f}")

            # Store results
            self.importance_results[dataset] = {
                'lr_coefs': lr_coefs,
                'rf_importances': rf_importances,
                'lr_score': lr_score,
                'rf_score': rf_score,
                'feature_names': ['PAM_norm', 'QAM_norm', 'SAM_norm']
            }

            # Print feature importances
            print(f"\n  Logistic Regression Coefficients:")
            for name, coef in zip(['PAM_norm', 'QAM_norm', 'SAM_norm'], lr_coefs):
                print(f"    {name}: {coef:.4f}")

            print(f"\n  Random Forest Feature Importances:")
            for name, imp in zip(['PAM_norm', 'QAM_norm', 'SAM_norm'], rf_importances):
                print(f"    {name}: {imp:.4f}")

        return self.importance_results

    def visualize_importance(self, output_dir='./week4_outputs'):
        """Create visualizations of feature importance"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n" + "-"*80)
        print("Creating feature importance visualizations...")
        print("-"*80)

        # Create bar plots for each dataset
        for dataset, results in self.importance_results.items():
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            # Logistic Regression coefficients
            axes[0].bar(results['feature_names'], results['lr_coefs'])
            axes[0].set_title(f'{dataset}: Logistic Regression Coefficients\n(Accuracy: {results["lr_score"]:.3f})')
            axes[0].set_ylabel('Coefficient Value')
            axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
            axes[0].tick_params(axis='x', rotation=45)

            # Random Forest importances
            axes[1].bar(results['feature_names'], results['rf_importances'], color='green')
            axes[1].set_title(f'{dataset}: Random Forest Feature Importances\n(Accuracy: {results["rf_score"]:.3f})')
            axes[1].set_ylabel('Importance')
            axes[1].tick_params(axis='x', rotation=45)

            plt.tight_layout()
            filename = f"{output_dir}/feature_importance_{dataset.lower()}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  - Saved {filename}")

        # Create combined comparison plot
        if len(self.importance_results) > 1:
            fig, axes = plt.subplots(2, 1, figsize=(12, 10))

            datasets = list(self.importance_results.keys())
            x = np.arange(len(results['feature_names']))
            width = 0.8 / len(datasets)

            # Logistic Regression comparison
            for i, dataset in enumerate(datasets):
                results = self.importance_results[dataset]
                axes[0].bar(x + i * width, results['lr_coefs'], width, label=dataset)
            axes[0].set_xlabel('Feature')
            axes[0].set_ylabel('Coefficient Value')
            axes[0].set_title('Logistic Regression Coefficients Across Datasets')
            axes[0].set_xticks(x + width * (len(datasets) - 1) / 2)
            axes[0].set_xticklabels(results['feature_names'])
            axes[0].legend()
            axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

            # Random Forest comparison
            for i, dataset in enumerate(datasets):
                results = self.importance_results[dataset]
                axes[1].bar(x + i * width, results['rf_importances'], width, label=dataset)
            axes[1].set_xlabel('Feature')
            axes[1].set_ylabel('Importance')
            axes[1].set_title('Random Forest Feature Importances Across Datasets')
            axes[1].set_xticks(x + width * (len(datasets) - 1) / 2)
            axes[1].set_xticklabels(results['feature_names'])
            axes[1].legend()

            plt.tight_layout()
            filename = f"{output_dir}/feature_importance_comparison.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  - Saved {filename}")

    def create_heatmaps(self, output_dir='./week4_outputs'):
        """Create heatmaps of head importance across layers"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n" + "-"*80)
        print("Creating correlation heatmaps...")
        print("-"*80)

        for dataset, corr_df in self.correlation_results.items():
            # Create pivot tables for heatmaps
            for metric in ['PAM_corr', 'QAM_corr', 'SAM_corr']:
                if metric not in corr_df.columns:
                    continue

                # Create pivot table
                pivot = corr_df.pivot(index='layer', columns='head', values=metric)

                # Create heatmap
                fig, ax = plt.subplots(figsize=(16, 8))
                sns.heatmap(pivot, annot=False, cmap='RdBu_r', center=0,
                           vmin=-0.5, vmax=0.5, cbar_kws={'label': 'Correlation'},
                           ax=ax)
                ax.set_title(f'{dataset}: {metric.replace("_", " ").title()} by Layer and Head')
                ax.set_xlabel('Head')
                ax.set_ylabel('Layer')

                plt.tight_layout()
                filename = f"{output_dir}/heatmap_{dataset.lower()}_{metric}.png"
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"  - Saved {filename}")


# ============================================================================
# RESEARCHER 3: CROSS-DATASET COMPARISON
# ============================================================================

class CrossDatasetComparison:
    """Compare head behavior across different datasets"""

    def __init__(self, correlation_results):
        """Initialize with correlation results from all datasets"""
        print("\n\n" + "="*80)
        print("RESEARCHER 3: CROSS-DATASET COMPARISON")
        print("="*80)

        self.correlation_results = correlation_results
        self.datasets = list(correlation_results.keys())

    def compute_similarity(self):
        """Compute similarity between datasets"""
        print("\n" + "-"*80)
        print("Computing cross-dataset similarity...")
        print("-"*80)

        if len(self.datasets) < 2:
            print("Need at least 2 datasets for comparison")
            return None

        # Create similarity matrix
        n_datasets = len(self.datasets)
        similarity_matrix = np.zeros((n_datasets, n_datasets))

        for i, dataset1 in enumerate(self.datasets):
            for j, dataset2 in enumerate(self.datasets):
                if i == j:
                    similarity_matrix[i, j] = 1.0
                else:
                    # Find common heads
                    df1 = self.correlation_results[dataset1]
                    df2 = self.correlation_results[dataset2]

                    # Merge on layer and head
                    merged = df1.merge(df2, on=['layer', 'head'], suffixes=('_1', '_2'))

                    if len(merged) == 0:
                        similarity_matrix[i, j] = 0.0
                        continue

                    # Compute cosine similarity of PAM correlations
                    vec1 = merged['PAM_corr_1'].fillna(0).values
                    vec2 = merged['PAM_corr_2'].fillna(0).values

                    # Cosine similarity = 1 - cosine distance
                    try:
                        sim = 1 - cosine(vec1, vec2)
                    except:
                        sim = 0.0

                    similarity_matrix[i, j] = sim

        self.similarity_matrix = similarity_matrix

        print("\nCosine Similarity Matrix (PAM correlations):")
        print(f"{'':15s} " + " ".join([f"{d:>12s}" for d in self.datasets]))
        for i, dataset in enumerate(self.datasets):
            row = " ".join([f"{similarity_matrix[i, j]:12.3f}" for j in range(n_datasets)])
            print(f"{dataset:15s} {row}")

        return similarity_matrix

    def identify_universal_heads(self, n_top=10):
        """Identify heads that are important across all datasets"""
        print("\n" + "-"*80)
        print(f"Identifying universal heads (top {n_top} across all datasets)...")
        print("-"*80)

        # Find heads that appear in top rankings across multiple datasets
        all_top_heads = {}

        for dataset, corr_df in self.correlation_results.items():
            # Get top PAM heads
            top_pam = set(corr_df.nlargest(n_top, 'PAM_corr')[['layer', 'head']].apply(tuple, axis=1))

            for head in top_pam:
                if head not in all_top_heads:
                    all_top_heads[head] = []
                all_top_heads[head].append(dataset)

        # Find heads appearing in all datasets
        universal_heads = [(head, datasets) for head, datasets in all_top_heads.items()
                          if len(datasets) == len(self.datasets)]

        print(f"\nUniversal PAM Heads (appear in top {n_top} of all datasets):")
        if universal_heads:
            for head, datasets in universal_heads:
                print(f"  Layer {head[0]}, Head {head[1]}: {datasets}")
        else:
            print("  No universal heads found")

        # Find dataset-specific heads
        dataset_specific = [(head, datasets) for head, datasets in all_top_heads.items()
                           if len(datasets) == 1]

        print(f"\nDataset-Specific PAM Heads (appear in top {n_top} of only one dataset):")
        for dataset in self.datasets:
            specific = [head for head, dsets in dataset_specific if dsets[0] == dataset]
            print(f"\n  {dataset}: {len(specific)} unique heads")
            for head in specific[:5]:  # Show top 5
                print(f"    Layer {head[0]}, Head {head[1]}")

        self.universal_heads = universal_heads
        self.dataset_specific = dataset_specific

        return universal_heads, dataset_specific

    def visualize_comparison(self, output_dir='./week4_outputs'):
        """Create cross-dataset comparison visualizations"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        print(f"\n" + "-"*80)
        print("Creating cross-dataset visualizations...")
        print("-"*80)

        # 1. Similarity matrix heatmap
        if hasattr(self, 'similarity_matrix'):
            plt.figure(figsize=(10, 8))
            sns.heatmap(self.similarity_matrix, annot=True, fmt='.3f',
                       xticklabels=self.datasets, yticklabels=self.datasets,
                       cmap='YlGnBu', vmin=0, vmax=1)
            plt.title('Cross-Dataset Similarity (PAM Correlations)')
            plt.tight_layout()
            filename = f"{output_dir}/cross_dataset_similarity.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  - Saved {filename}")

        # 2. 3D scatter plot of correlations
        if len(self.datasets) >= 2:
            from mpl_toolkits.mplot3d import Axes3D

            fig = plt.figure(figsize=(14, 10))

            for idx, dataset in enumerate(self.datasets):
                ax = fig.add_subplot(2, 2, idx+1, projection='3d')

                corr_df = self.correlation_results[dataset]

                # Filter out NaN values
                plot_data = corr_df.dropna(subset=['PAM_corr', 'QAM_corr', 'SAM_corr'])

                scatter = ax.scatter(plot_data['PAM_corr'],
                                    plot_data['QAM_corr'],
                                    plot_data['SAM_corr'],
                                    c=plot_data['layer'],
                                    cmap='viridis',
                                    alpha=0.6)

                ax.set_xlabel('PAM Correlation')
                ax.set_ylabel('QAM Correlation')
                ax.set_zlabel('SAM Correlation')
                ax.set_title(f'{dataset}')
                plt.colorbar(scatter, ax=ax, label='Layer')

            plt.suptitle('3D Scatter: PAM vs QAM vs SAM Correlations by Dataset')
            plt.tight_layout()
            filename = f"{output_dir}/cross_dataset_3d_scatter.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  - Saved {filename}")

        # 3. Per-layer comparison across datasets
        fig, axes = plt.subplots(3, 1, figsize=(14, 12))

        for metric_idx, metric in enumerate(['PAM_corr', 'QAM_corr', 'SAM_corr']):
            ax = axes[metric_idx]

            for dataset in self.datasets:
                corr_df = self.correlation_results[dataset]

                # Aggregate by layer
                layer_stats = corr_df.groupby('layer')[metric].agg(['mean', 'std']).reset_index()

                ax.errorbar(layer_stats['layer'], layer_stats['mean'],
                           yerr=layer_stats['std'], label=dataset,
                           marker='o', capsize=5, alpha=0.7)

            ax.set_xlabel('Layer')
            ax.set_ylabel(f'{metric.replace("_", " ").title()}')
            ax.set_title(f'{metric.replace("_", " ").title()} by Layer Across Datasets')
            ax.legend()
            ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        filename = f"{output_dir}/per_layer_comparison.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  - Saved {filename}")

    def write_interpretation(self, output_dir='./week4_outputs'):
        """Write interpretation of cross-dataset findings"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        filename = f"{output_dir}/cross_dataset_interpretation.txt"

        with open(filename, 'w') as f:
            f.write("="*80 + "\n")
            f.write("CROSS-DATASET COMPARISON: INTERPRETATION\n")
            f.write("="*80 + "\n\n")

            f.write("DATASET SIMILARITY:\n")
            f.write("-"*80 + "\n")
            if hasattr(self, 'similarity_matrix'):
                for i, dataset1 in enumerate(self.datasets):
                    for j, dataset2 in enumerate(self.datasets):
                        if i < j:
                            sim = self.similarity_matrix[i, j]
                            f.write(f"{dataset1} vs {dataset2}: {sim:.3f}\n")
            f.write("\n")

            f.write("KEY FINDINGS:\n")
            f.write("-"*80 + "\n\n")

            # Analyze each dataset's characteristics
            for dataset, corr_df in self.correlation_results.items():
                f.write(f"{dataset.upper()}:\n")

                # Compute statistics
                pam_mean = corr_df['PAM_corr'].mean()
                qam_mean = corr_df['QAM_corr'].mean()
                sam_mean = corr_df['SAM_corr'].mean()

                pam_std = corr_df['PAM_corr'].std()

                f.write(f"  - Mean PAM correlation: {pam_mean:.4f} (std: {pam_std:.4f})\n")
                f.write(f"  - Mean QAM correlation: {qam_mean:.4f}\n")
                f.write(f"  - Mean SAM correlation: {sam_mean:.4f}\n")

                # Interpret
                if abs(pam_mean) < 0.05:
                    f.write(f"  - Interpretation: Very weak correlation signal in {dataset}\n")
                    f.write(f"    Possible reasons: uniform behavior across prompts, insufficient variance\n")
                elif pam_mean > 0.1:
                    f.write(f"  - Interpretation: Strong prompt-following signal detected\n")
                    f.write(f"    This dataset shows clear attention → behavior relationship\n")
                elif pam_mean < -0.1:
                    f.write(f"  - Interpretation: Negative correlation - higher prompt attention → worse behavior\n")
                    f.write(f"    This is unexpected and warrants investigation\n")

                f.write("\n")

            f.write("\n" + "="*80 + "\n")
            f.write("RECOMMENDATIONS FOR WEEK 5:\n")
            f.write("="*80 + "\n\n")

            # Identify which dataset to focus on
            best_dataset = None
            best_signal = 0

            for dataset, corr_df in self.correlation_results.items():
                pam_mean = abs(corr_df['PAM_corr'].mean())
                if pam_mean > best_signal:
                    best_signal = pam_mean
                    best_dataset = dataset

            if best_dataset:
                f.write(f"Focus causal experiments on: {best_dataset}\n")
                f.write(f"  - This dataset shows the strongest correlation signal ({best_signal:.4f})\n")
                f.write(f"  - Causal interventions are more likely to produce measurable effects\n\n")

            # Suggest specific heads for intervention
            if best_dataset:
                corr_df = self.correlation_results[best_dataset]
                top_pam = corr_df.nlargest(5, 'PAM_corr')[['layer', 'head', 'PAM_corr']]

                f.write("Suggested heads for PAM boost:\n")
                for _, row in top_pam.iterrows():
                    f.write(f"  - Layer {int(row['layer'])}, Head {int(row['head'])} (corr: {row['PAM_corr']:.4f})\n")

                f.write("\n")

                bottom_sam = corr_df.nsmallest(5, 'SAM_corr')[['layer', 'head', 'SAM_corr']]
                f.write("Suggested heads for SAM suppression:\n")
                for _, row in bottom_sam.iterrows():
                    f.write(f"  - Layer {int(row['layer'])}, Head {int(row['head'])} (corr: {row['SAM_corr']:.4f})\n")

        print(f"  - Saved {filename}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_week4_analysis(data_path, output_dir='./week4_outputs'):
    """
    Run complete Week 4 analysis pipeline

    Args:
        data_path: Path to all_metrics_NORMALIZED.parquet
        output_dir: Directory for output files
    """
    print("\n" + "="*80)
    print("WEEK 4: COMPLETE HEAD-LEVEL ANALYSIS PIPELINE")
    print("="*80 + "\n")

    # RESEARCHER 1: Compute correlations
    researcher1 = HeadLevelCorrelations(data_path)
    correlation_results = researcher1.compute_correlations()
    top_heads = researcher1.identify_top_heads(n_top=10)
    researcher1.save_results(output_dir)

    # RESEARCHER 2: Feature importance
    researcher2 = FeatureImportanceAnalysis(researcher1.df, correlation_results)
    importance_results = researcher2.compute_feature_importance()
    researcher2.visualize_importance(output_dir)
    researcher2.create_heatmaps(output_dir)

    # RESEARCHER 3: Cross-dataset comparison
    researcher3 = CrossDatasetComparison(correlation_results)
    similarity = researcher3.compute_similarity()
    universal, specific = researcher3.identify_universal_heads(n_top=10)
    researcher3.visualize_comparison(output_dir)
    researcher3.write_interpretation(output_dir)

    print("\n" + "="*80)
    print("WEEK 4 ANALYSIS COMPLETE!")
    print("="*80)
    print(f"\nAll outputs saved to: {output_dir}")
    print("\nGenerated files:")
    print("  - head_corr_[dataset].csv: Per-head correlation tables")
    print("  - top_heads_summary.txt: Top heads for each metric")
    print("  - feature_importance_*.png: Feature importance plots")
    print("  - heatmap_*.png: Correlation heatmaps")
    print("  - cross_dataset_*.png: Cross-dataset comparison plots")
    print("  - cross_dataset_interpretation.txt: Analysis interpretation")
    print("\n" + "="*80 + "\n")

    return {
        'correlation_results': correlation_results,
        'top_heads': top_heads,
        'importance_results': importance_results,
        'universal_heads': universal,
        'dataset_specific': specific
    }


# For Google Colab, don't use command line arguments
# Just call the function directly with your data path

In [7]:
"""
Simple diagnostic script to test if Week 4 is working
Run this in Colab to see what's happening
"""

import pandas as pd
import numpy as np

# Test 1: Can we load the data?
print("="*80)
print("TEST 1: Loading data")
print("="*80)

try:
    df = pd.read_parquet('/content/all_metrics_NORMALIZED.parquet')
    print(f"✅ SUCCESS: Loaded {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types:")
    print(df.dtypes)
except Exception as e:
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

# Test 2: Check required columns
print("\n" + "="*80)
print("TEST 2: Checking required columns")
print("="*80)

required_cols = ['layer', 'head', 'PAM_norm', 'QAM_norm', 'SAM_norm', 'behavior_label', 'dataset']
for col in required_cols:
    if col in df.columns:
        print(f"✅ {col}: Found")
        print(f"   - Unique values: {df[col].nunique()}")
        print(f"   - Missing: {df[col].isna().sum()}")
    else:
        print(f"❌ {col}: MISSING")

# Test 3: Check data quality
print("\n" + "="*80)
print("TEST 3: Data quality check")
print("="*80)

print("\nDatasets present:")
if 'dataset' in df.columns:
    print(df['dataset'].value_counts())

print("\nBehavior labels:")
if 'behavior_label' in df.columns:
    print(df['behavior_label'].value_counts())

print("\nHead values:")
if 'head' in df.columns:
    print(f"Unique heads: {sorted(df['head'].unique())}")
    print(f"Has 'avg': {'avg' in df['head'].values}")

# Test 4: Try running a simple correlation
print("\n" + "="*80)
print("TEST 4: Simple correlation test")
print("="*80)

try:
    # Clean data like the script does
    df_clean = df.copy()
    df_clean = df_clean.dropna(subset=['behavior_label'])
    df_clean = df_clean[df_clean['head'] != 'avg']
    df_clean['head'] = pd.to_numeric(df_clean['head'], errors='coerce')
    df_clean = df_clean.dropna(subset=['head'])
    df_clean['head'] = df_clean['head'].astype(int)

    print(f"After cleaning: {len(df_clean)} rows")

    # Try computing one correlation
    if 'sharegpt' in df_clean['dataset'].values:
        test_df = df_clean[df_clean['dataset'] == 'sharegpt']
        test_group = test_df.groupby(['layer', 'head']).first()
        print(f"ShareGPT grouped: {len(test_group)} head groups")

        if len(test_group) > 0:
            from scipy.stats import pearsonr
            first_group = test_df[test_df['layer'] == test_df['layer'].iloc[0]]
            first_group = first_group[first_group['head'] == first_group['head'].iloc[0]]

            if len(first_group) >= 2:
                corr, pval = pearsonr(first_group['PAM_norm'], first_group['behavior_label'])
                print(f"✅ Test correlation successful: {corr:.4f} (p={pval:.4f})")
            else:
                print(f"⚠️ Not enough samples in first group ({len(first_group)})")
    else:
        print("⚠️ ShareGPT not in datasets")

except Exception as e:
    print(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()

# Test 5: Try importing the script
print("\n" + "="*80)
print("TEST 5: Can we import the script?")
print("="*80)

try:
    from week4_colab_version import run_week4_analysis
    print("✅ Successfully imported run_week4_analysis")
except Exception as e:
    print(f"❌ ERROR importing: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("DIAGNOSTIC COMPLETE")
print("="*80)
print("\nIf all tests pass, the script should work.")
print("If any test fails, that's the problem to fix!")

TEST 1: Loading data
✅ SUCCESS: Loaded 280896 rows
Columns: ['id', 'dataset', 'layer', 'head', 'PAM_norm', 'QAM_norm', 'SAM_norm']

First few rows:
             id dataset  layer head  PAM_norm  QAM_norm  SAM_norm
0  alpaca:22052  alpaca      0  avg  0.756447  0.107926  0.135627
1  alpaca:22052  alpaca      0    0  0.613634  0.199925  0.186441
2  alpaca:22052  alpaca      0    1  0.000226  0.143504  0.856270
3  alpaca:22052  alpaca      0    2  0.666825  0.176040  0.157134
4  alpaca:22052  alpaca      0    3  0.808819  0.093579  0.097602

Data types:
id           object
dataset      object
layer         int64
head         object
PAM_norm    float64
QAM_norm    float64
SAM_norm    float64
dtype: object

TEST 2: Checking required columns
✅ layer: Found
   - Unique values: 32
   - Missing: 0
✅ head: Found
   - Unique values: 33
   - Missing: 0
✅ PAM_norm: Found
   - Unique values: 280896
   - Missing: 0
✅ QAM_norm: Found
   - Unique values: 280896
   - Missing: 0
✅ SAM_norm: Found
   - Un

Traceback (most recent call last):
  File "/tmp/ipython-input-3266556411.py", line 67, in <cell line: 0>
    df_clean = df_clean.dropna(subset=['behavior_label'])
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 6670, in dropna
    raise KeyError(np.array(subset)[check].tolist())
KeyError: ['behavior_label']
Traceback (most recent call last):
  File "/tmp/ipython-input-3266556411.py", line 105, in <cell line: 0>
    from week4_colab_version import run_week4_analysis
ModuleNotFoundError: No module named 'week4_colab_version'
